# Обучение модели на корпусе данных RuSTS

В данной работе была обучена модель multilingual-e5-small на корпусе данных RuSTS двумя стратегиями:
1) Baseline - стандартная форма обучения в 3 эпохи
2) Curriculum - обучение с варьированием сложности обучающих примеров

Импортируем необходимые библиотеки:

In [1]:
from datasets import load_dataset
import pandas as pd
import requests
from io import StringIO
from tqdm import tqdm
from collections import defaultdict
import builtins

In [2]:
import torch
import transformers
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader


In [3]:
from datetime import datetime

Загружаем корпус данных:

In [4]:
dataset = load_dataset("mteb/RuSTSBenchmarkSTS", split="train")
print(" Загружен mteb/RuSTSBenchmarkSTS")

 Загружен mteb/RuSTSBenchmarkSTS


In [5]:
train_examples_data = []

for row in tqdm(dataset):
    s1 = row.get('sentence1') or row.get('text_1')
    s2 = row.get('sentence2') or row.get('text_2')
    score = row.get('similarity_score') or row.get('score') or row.get('label')

    if score is None:
        continue
        
    try:
        score = float(score)
    except (ValueError, TypeError):
        continue
    
    threshold = 3.0 if score > 1.0 else 0.6
    
    if score >= threshold:
        train_examples_data.append({
            'query': s1,
            'passage': s2,
            'original_score': score
        })

df = pd.DataFrame(train_examples_data)
print(f"  Всего примеров для обучения: {len(df)}")

100%|███████████████████████████████████████████████████████████████████████████| 5224/5224 [00:00<00:00, 52993.71it/s]

  Всего примеров для обучения: 3147


Вычисляем сложность примеров для Cirriculum learning:

In [6]:
df['query_len'] = df['query'].apply(lambda x: len(str(x).split()))
df['passage_len'] = df['passage'].apply(lambda x: len(str(x).split()))

def calculate_difficulty(row):
    # Преобразуем в строки на случай нетекстовых данных
    q_text = str(row['query']).lower()
    p_text = str(row['passage']).lower()
    
    query_words = set(q_text.split())
    passage_words = set(p_text.split())
    
    intersection = len(query_words & passage_words)
    union = len(query_words | passage_words)
    lexical_overlap = intersection / union if union > 0 else 0
    norm_passage_len = min(row['passage_len'] / 50, 1.0) 
    norm_query_len = min(row['query_len'] / 50, 1.0)
    
    difficulty = (norm_passage_len * 0.4 + 
                  norm_query_len * 0.2 + 
                  (1 - lexical_overlap) * 0.4)
    
    return difficulty

In [7]:
tqdm.pandas(desc="Calculating difficulty")
df['difficulty'] = df.apply(calculate_difficulty, axis=1)

print(f"Сложность:")
print(df['difficulty'].describe())

Сложность:
count    3147.000000
mean        0.389306
std         0.109799
min         0.024000
25%         0.327385
50%         0.407273
75%         0.463731
max         0.717725
Name: difficulty, dtype: float64


In [8]:
df_sorted = df.sort_values('difficulty').reset_index(drop=True)

df['difficulty'] = df.apply(calculate_difficulty, axis=1)
n_stages = 4
stage_size = len(df_sorted) // n_stages
stages = {}

print("\nCurriculum Stages:")
for i in range(n_stages):
    start = i * stage_size
    end = (i + 1) * stage_size if i < n_stages - 1 else len(df_sorted)
    
    name = f'stage_{i+1}'
    stages[name] = df_sorted.iloc[start:end]
    
    avg_diff = stages[name]['difficulty'].mean()
    print(f"  Stage {i+1}: {len(stages[name])} примеров (Avg Difficulty: {avg_diff:.3f})")


Curriculum Stages:
  Stage 1: 786 примеров (Avg Difficulty: 0.239)
  Stage 2: 786 примеров (Avg Difficulty: 0.371)
  Stage 3: 786 примеров (Avg Difficulty: 0.436)
  Stage 4: 789 примеров (Avg Difficulty: 0.511)


In [9]:
model_name = 'intfloat/multilingual-e5-small'
batch_size = 16
num_epochs_baseline = 3
num_epochs_curriculum = 2

val_df = df.sample(frac=0.1, random_state=42)
val_examples = [
    InputExample(texts=[f"query: {q}", f"passage: {p}"], label=1.0) 
    for q, p in zip(val_df['query'], val_df['passage'])
]
evaluator = EmbeddingSimilarityEvaluator.from_input_examples(
    val_examples, name='rusts-val'
)

In [10]:
print("ОБУЧЕНИЕ BASELINE")

baseline_model = SentenceTransformer(model_name)
train_loss = losses.MultipleNegativesRankingLoss(baseline_model)

train_data_baseline = []
for _, row in df.iterrows():
    train_data_baseline.append(InputExample(texts=[f"query: {row['query']}", f"passage: {row['passage']}"]))

train_dataloader = DataLoader(train_data_baseline, shuffle=True, batch_size=batch_size)

baseline_path = f'models/rusts_baseline_{datetime.now().strftime("%H%M")}'

baseline_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=num_epochs_baseline,
    evaluator=evaluator,
    output_path=baseline_path,
    save_best_model=True,
    show_progress_bar=True
)
print(f" Baseline сохранен: {baseline_path}")

ОБУЧЕНИЕ BASELINE


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Rusts-val Pearson Cosine,Rusts-val Spearman Cosine
197,No log,No log,nan,nan
394,No log,No log,nan,nan
500,0.597600,No Log,No Log,No Log
591,0.597600,No log,nan,nan


C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:196: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman, _ = spearmanr(labels, scores)


 Baseline сохранен: models/rusts_baseline_0913


In [11]:
import os

In [12]:
print("5. ОБУЧЕНИЕ CURRICULUM")

curriculum_model = SentenceTransformer(model_name)
train_loss_curr = losses.MultipleNegativesRankingLoss(curriculum_model)
curriculum_path_base = f'models/rusts_curriculum_{datetime.now().strftime("%H%M")}'

for i, (name, stage_df) in enumerate(stages.items()):
    print(f"\n--- Stage {i+1}: {name} ---")
    
    stage_data = []
    for _, row in stage_df.iterrows():
        stage_data.append(InputExample(texts=[f"query: {row['query']}", f"passage: {row['passage']}"]))
        
    stage_loader = DataLoader(stage_data, shuffle=True, batch_size=batch_size)
    
    stage_path = os.path.join(curriculum_path_base, name)
    
    curriculum_model.fit(
        train_objectives=[(stage_loader, train_loss_curr)],
        epochs=num_epochs_curriculum,
        evaluator=evaluator,
        output_path=stage_path,
        save_best_model=True,
        show_progress_bar=True
    )

final_path = os.path.join(curriculum_path_base, "final_model")
curriculum_model.save(final_path)
print(f"\n✅ Curriculum сохранен: {final_path}")

5. ОБУЧЕНИЕ CURRICULUM

--- Stage 1: stage_1 ---


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Rusts-val Pearson Cosine,Rusts-val Spearman Cosine
50,No log,No log,nan,nan
100,No log,No log,nan,nan


C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:196: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman, _ = spearmanr(labels, scores)



--- Stage 2: stage_2 ---


...

C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:196: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman, _ = spearmanr(labels, scores)



--- Stage 4: stage_4 ---


Step,Training Loss,Validation Loss,Rusts-val Pearson Cosine,Rusts-val Spearman Cosine
50,No log,No log,nan,nan
100,No log,No log,nan,nan


C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:195: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_pearson, _ = pearsonr(labels, scores)
C:\Users\Александр\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\evaluation\EmbeddingSimilarityEvaluator.py:196: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  eval_spearman, _ = spearmanr(labels, scores)



✅ Curriculum сохранен: models/rusts_curriculum_0914\final_model
